# **`Spotify Songs Data Cleaning Script`**


In [2]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import re
from io import StringIO
# Load the data
df=pd.read_csv('/content/Most Streamed Spotify Songs 2024 (1).csv',
                       encoding='latin-1')
# Display basic info about the dataset
print("=" * 60)
print("INITIAL DATA OVERVIEW")
print("=" * 60)
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")

print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

INITIAL DATA OVERVIEW
Dataset shape: (4600, 29)

Column names:
  1. Track
  2. Album Name
  3. Artist
  4. Release Date
  5. ISRC
  6. All Time Rank
  7. Track Score
  8. Spotify Streams
  9. Spotify Playlist Count
  10. Spotify Playlist Reach
  11. Spotify Popularity
  12. YouTube Views
  13. YouTube Likes
  14. TikTok Posts
  15. TikTok Likes
  16. TikTok Views
  17. YouTube Playlist Reach
  18. Apple Music Playlist Count
  19. AirPlay Spins
  20. SiriusXM Spins
  21. Deezer Playlist Count
  22. Deezer Playlist Reach
  23. Amazon Playlist Count
  24. Pandora Streams
  25. Pandora Track Stations
  26. Soundcloud Streams
  27. Shazam Counts
  28. TIDAL Popularity
  29. Explicit Track

Data types:
Track                          object
Album Name                     object
Artist                         object
Release Date                   object
ISRC                           object
All Time Rank                  object
Track Score                   float64
Spotify Streams             

,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0
3,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,USSM12209777,4,444.9,"2,031,280,633","269,802","136,569,078",...,"2,182",264.0,"24,684,248",210.0,"190,260,277","203,384",NaN,"11,822,942",NaN,0
4,Houdini,Houdini,Eminem,5/31/2024,USUG12403398,5,423.3,"107,034,922","7,223","151,469,874",...,1,82.0,"17,660,624",105.0,"4,493,884","7,006","207,179","457,017",NaN,1


## ***`Handling Missing Values`***

In [4]:
print("=" * 60)
print("STEP 1: HANDLING MISSING VALUES")
print("=" * 60)
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing_Count': missing_values.values,
    'Missing_Percentage': missing_percentage.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(f"\nColumns with missing values:")
if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print("No missing values found!")
# Handling missing values based on column types
for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':  # Categorical/text columns
            # For text columns, fill with 'Unknown' or mode
            if col in ['Track', 'Album Name', 'Artist', 'ISRC']:
                df[col] = df[col].fillna('Unknown')
            else:
                # For other text columns, fill with most frequent value
                df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown')
        else:  # Numeric columns
            # For numeric columns, fill with 0 or median based on context
            if col in ['TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach']:
                df[col] = df[col].fillna(0)  # These might be genuinely 0
            else:
                # For other numeric columns, use median to avoid outlier influence
                df[col] = df[col].fillna(df[col].median())

print(f"\n Missing values handled")

STEP 1: HANDLING MISSING VALUES

Columns with missing values:
          Column  Missing_Count  Missing_Percentage
TIDAL Popularity           4600               100.0

 Missing values handled


## ***`Fix encoding issues and clean text fields`***

In [5]:
print("=" * 60)
print("STEP 2: CLEANING TEXT FIELDS")
print("=" * 60)
def clean_text(text):
    """Clean text by removing special characters and fixing encoding"""
    if pd.isna(text) or text == 'Unknown':
        return text
    text = str(text)# Convert to string
    text = re.sub(r'[ïýŋ―�]', '', text)  # Remove problematic characters
    text = ' '.join(text.split())
    return text
# Apply cleaning to text columns
text_columns = df.select_dtypes(include=['object']).columns
for col in text_columns:
    df[col] = df[col].apply(clean_text)
# Special handling for Track and Artist columns
print(f"\nCleaned {len(text_columns)} text columns")
print("Sample of cleaned Track names:")
for i, track in enumerate(df['Track'].head(10)):
    print(f"  {i+1}. {track}")

STEP 2: CLEANING TEXT FIELDS

Cleaned 22 text columns
Sample of cleaned Track names:
  1. MILLION DOLLAR BABY
  2. Not Like Us
  3. i like the way you kiss me
  4. Flowers
  5. Houdini
  6. Lovin On Me
  7. Beautiful Things
  8. Gata Only
  9. Danza Kuduro - Cover
  10. BAND4BAND (feat. Lil Baby)


## ***`Convert data types and clean numeric fields`***

In [6]:
print("=" * 60)
print("STEP 3: CONVERTING DATA TYPES")
print("=" * 60)
def clean_numeric(value):
    """Convert string numbers with commas to float"""
    if pd.isna(value) or value == '' or value == 'Unknown':
        return np.nan
    try:
        # Remove commas and convert to float
        return float(str(value).replace(',', '').strip())
    except:
        return np.nan
# Columns that should be numeric (based on the data)
numeric_columns = [
    'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count',
    'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes',
    'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach',
    'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins',
    'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count',
    'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams',
    'Shazam Counts', 'TIDAL Popularity'
]
# Convert numeric columns
conversion_stats = {}
for col in numeric_columns:
    if col in df.columns:
        original = df[col].copy()
        df[col] = df[col].apply(clean_numeric)
         # Track conversion stats
        converted = df[col].notna().sum()
        total = len(df)
        conversion_stats[col] = f"{converted}/{total} ({converted/total*100:.1f}%)"
print("Numeric column conversion success rate:")
for col, stats in list(conversion_stats.items())[:10]:  # Show first 10
    print(f"  {col}: {stats}")
print("  ... (and more)")
# Convert Release Date to datetime
print(f"\nConverting Release Date to datetime...")
df['Release Date'] = pd.to_datetime(df['Release Date'], errors='coerce')
print(f"  Successfully converted: {df['Release Date'].notna().sum()}/{len(df)} rows")

STEP 3: CONVERTING DATA TYPES
Numeric column conversion success rate:
  All Time Rank: 4600/4600 (100.0%)
  Track Score: 4600/4600 (100.0%)
  Spotify Streams: 4600/4600 (100.0%)
  Spotify Playlist Count: 4600/4600 (100.0%)
  Spotify Playlist Reach: 4600/4600 (100.0%)
  Spotify Popularity: 4600/4600 (100.0%)
  YouTube Views: 4600/4600 (100.0%)
  YouTube Likes: 4600/4600 (100.0%)
  TikTok Posts: 4600/4600 (100.0%)
  TikTok Likes: 4600/4600 (100.0%)
  ... (and more)

Converting Release Date to datetime...
  Successfully converted: 4600/4600 rows


 ## ***`Handle the Explicit Track column`***

In [7]:
print("=" * 60)
print("STEP 4: CLEANING EXPLICIT TRACK COLUMN")
print("=" * 60)
# Check the Explicit Track column
print(f"Unique values in 'Explicit Track': {df['Explicit Track'].unique()}")
# Convert to boolean (0/1 or True/False)
# Based on the data, it appears 1 = explicit, 0 = not explicit
df['Explicit'] = df['Explicit Track'].apply(lambda x: 1 if x == 1 or x == '1' else 0)
# Drop the original column
df = df.drop('Explicit Track', axis=1)
print(f"\nExplicit track distribution:")
print(df['Explicit'].value_counts())
print(f"  Explicit (1): {df['Explicit'].sum()} tracks")
print(f"  Clean (0): {len(df) - df['Explicit'].sum()} tracks")

STEP 4: CLEANING EXPLICIT TRACK COLUMN
Unique values in 'Explicit Track': [0 1]

Explicit track distribution:
Explicit
0    2949
1    1651
Name: count, dtype: int64
  Explicit (1): 1651 tracks
  Clean (0): 2949 tracks


## ***`Remove duplicates`***

In [8]:
print("=" * 60)
print("STEP 5: REMOVING DUPLICATES")
print("=" * 60)
# Check for duplicates based on key identifiers
# A track might be considered duplicate if it has same Track, Artist, and ISRC
duplicate_cols = ['Track', 'Artist', 'ISRC']
duplicates = df.duplicated(subset=duplicate_cols, keep='first')
print(f"Found {duplicates.sum()} duplicate tracks")
if duplicates.sum() > 0:
    # Display duplicate examples
    duplicate_tracks = df[duplicates][duplicate_cols].head()
    print("\nExample duplicates:")
    print(duplicate_tracks.to_string(index=False))
    # Remove duplicates
    df = df.drop_duplicates(subset=duplicate_cols, keep='first')
    print(f"\n✓ Removed duplicates. New shape: {df.shape}")
else:
    print("✓ No duplicates found")

STEP 5: REMOVING DUPLICATES
Found 2 duplicate tracks

Example duplicates:
           Track        Artist         ISRC
Tennessee Orange Megan Moroney TCAGJ2289254
          Dembow   Danny Ocean USWL11700269

✓ Removed duplicates. New shape: (4598, 29)


## ***`Handle outliers and validate data`***

In [9]:
print("=" * 60)
print("STEP 6: HANDLING OUTLIERS")
print("=" * 60)
# Function to detect outliers using IQR method
def detect_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR  # Using 3*IQR for extreme outliers
    upper_bound = Q3 + 3 * IQR
    return (series < lower_bound) | (series > upper_bound)
# Check for outliers in key numeric columns
key_metrics = ['Spotify Streams', 'YouTube Views', 'TikTok Views', 'Track Score']
outlier_counts = {}
print("Outlier analysis (extreme values > 3*IQR):")
for col in key_metrics:
    if col in df.columns and df[col].dtype in ['float64', 'int64']:
        outliers = detect_outliers(df[col].dropna())
        outlier_count = outliers.sum()
        outlier_counts[col] = outlier_count
        print(f"  {col}: {outlier_count} outliers ({outlier_count/len(df)*100:.1f}%)")
# For this dataset, outliers might be legitimate (viral hits)
# We'll cap extreme outliers at the 99.9th percentile to be safe
print("\nCapping extreme outliers at 99.9th percentile...")
for col in key_metrics:
    if col in df.columns and df[col].dtype in ['float64', 'int64']:
        p999 = df[col].quantile(0.999)
        original_max = df[col].max()
        df[col] = df[col].clip(upper=p999)
        print(f"  {col}: capped from {original_max:,.0f} to {p999:,.0f}")

STEP 6: HANDLING OUTLIERS
Outlier analysis (extreme values > 3*IQR):
  Spotify Streams: 35 outliers (0.8%)
  YouTube Views: 232 outliers (5.0%)
  TikTok Views: 364 outliers (7.9%)
  Track Score: 197 outliers (4.3%)

Capping extreme outliers at 99.9th percentile...
  Spotify Streams: capped from 4,281,468,720 to 3,386,428,393
  YouTube Views: capped from 16,322,756,555 to 6,391,486,496
  TikTok Views: capped from 233,232,311,463 to 33,565,490,087
  Track Score: capped from 725 to 415


## ***`Extract additional features from Release Date`***

In [10]:
print("=" * 60)
print("STEP 7: EXTRACTING DATE FEATURES")
print("=" * 60)
# Create new date-based features
df['Release_Year'] = df['Release Date'].dt.year
df['Release_Month'] = df['Release Date'].dt.month
df['Release_Day'] = df['Release Date'].dt.day
df['Release_DayOfWeek'] = df['Release Date'].dt.dayofweek  # Monday=0, Sunday=6
df['Release_Quarter'] = df['Release Date'].dt.quarter
# Map day of week to name
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
             4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df['Release_DayName'] = df['Release_DayOfWeek'].map(day_names)

print(f"Added {len(['Release_Year', 'Release_Month', 'Release_Day', 'Release_DayOfWeek', 'Release_Quarter', 'Release_DayName'])} new date features")
print("\nSample of new features:")
df[['Release Date', 'Release_Year', 'Release_Month', 'Release_DayName']].head(10)

STEP 7: EXTRACTING DATE FEATURES
Added 6 new date features

Sample of new features:


,Release Date,Release_Year,Release_Month,Release_DayName
0,2024-04-26,2024,4,Friday
1,2024-05-04,2024,5,Saturday
2,2024-03-19,2024,3,Tuesday
3,2023-01-12,2023,1,Thursday
4,2024-05-31,2024,5,Friday
5,2023-11-10,2023,11,Friday
6,2024-01-18,2024,1,Thursday
7,2024-02-02,2024,2,Friday
8,2024-06-09,2024,6,Sunday
9,2024-05-23,2024,5,Thursday


## ***`Create derived metrics for analysis`***

In [11]:
print("=" * 60)
print("STEP 8: CREATING DERIVED METRICS")
print("=" * 60)
# Calculate total platform presence
platform_cols = ['Spotify Playlist Count', 'Apple Music Playlist Count',
                 'Deezer Playlist Count', 'Amazon Playlist Count']
df['Total_Playlist_Count'] = df[platform_cols].sum(axis=1)
# Calculate engagement ratios (where applicable)
# YouTube like ratio
df['YouTube_Like_Ratio'] = df['YouTube Likes'] / df['YouTube Views'].replace(0, np.nan)
df['YouTube_Like_Ratio'] = df['YouTube_Like_Ratio'].clip(0, 1)  # Cap at 1 (100%)
# TikTok engagement (likes per view)
df['TikTok_Like_Ratio'] = df['TikTok Likes'] / df['TikTok Views'].replace(0, np.nan)
df['TikTok_Like_Ratio'] = df['TikTok_Like_Ratio'].clip(0, 1)
# Spotify popularity score (normalized)
df['Spotify_Popularity_Norm'] = df['Spotify Popularity'] / 100
# Create platform success score (simple average of normalized metrics)
df['Platform_Success_Score'] = (
    df['Spotify_Popularity_Norm'] * 0.3 +
    (df['YouTube_Like_Ratio'].fillna(0)) * 0.2 +
    (df['TikTok_Like_Ratio'].fillna(0)) * 0.2 +
    (df['Track Score'] / df['Track Score'].max()) * 0.3
)
print(f"Added {4} new derived metrics")
print("\nDerived metrics description:")
print("  Total_Playlist_Count: Sum of playlist appearances across platforms")
print("  YouTube_Like_Ratio: Ratio of likes to views on YouTube")
print("  TikTok_Like_Ratio: Ratio of likes to views on TikTok")
print("  Platform_Success_Score: Composite score of platform performance (0-1)")

STEP 8: CREATING DERIVED METRICS
Added 4 new derived metrics

Derived metrics description:
  Total_Playlist_Count: Sum of playlist appearances across platforms
  YouTube_Like_Ratio: Ratio of likes to views on YouTube
  TikTok_Like_Ratio: Ratio of likes to views on TikTok
  Platform_Success_Score: Composite score of platform performance (0-1)


## ***`Final data validation and summary`***

In [12]:
print("=" * 60)
print("STEP 9: FINAL DATA SUMMARY")
print("=" * 60)
print(f"Final dataset shape: {df.shape}")
print(f"Number of unique tracks: {df['Track'].nunique()}")
print(f"Number of unique artists: {df['Artist'].nunique()}")
print(f"Date range: {df['Release Date'].min()} to {df['Release Date'].max()}")
print("\n" + "=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)
# Check for any remaining issues
quality_checks = {
    "Missing values": df.isnull().sum().sum(),
    "Duplicate tracks": df.duplicated(subset=['Track', 'Artist']).sum(),
    "Invalid dates": df['Release Date'].isna().sum(),
    "Zero streams": (df['Spotify Streams'] == 0).sum(),
}
for check, value in quality_checks.items():
    status = "✓ GOOD" if value == 0 else f"⚠ WARNING: {value}"
    print(f"  {check}: {status}")
print("\n" + "=" * 60)
print("DATA TYPES AFTER CLEANING")
print("=" * 60)
print(df.dtypes)
print("\n" + "=" * 60)
print("FIRST 5 ROWS OF CLEANED DATA")
print("=" * 60)
df.head()

STEP 9: FINAL DATA SUMMARY
Final dataset shape: (4598, 40)
Number of unique tracks: 4336
Number of unique artists: 1989
Date range: 1987-07-21 00:00:00 to 2024-06-14 00:00:00

DATA QUALITY CHECK
  Missing values: ⚠ WARNING: 4598
  Duplicate tracks: ⚠ WARNING: 123
  Invalid dates: ✓ GOOD
  Zero streams: ✓ GOOD

DATA TYPES AFTER CLEANING
Track                                 object
Album Name                            object
Artist                                object
Release Date                  datetime64[ns]
ISRC                                  object
All Time Rank                        float64
Track Score                          float64
Spotify Streams                      float64
Spotify Playlist Count               float64
Spotify Playlist Reach               float64
Spotify Popularity                   float64
YouTube Views                        float64
YouTube Likes                        float64
TikTok Posts                         float64
TikTok Likes                    

,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,Release_Month,Release_Day,Release_DayOfWeek,Release_Quarter,Release_DayName,Total_Playlist_Count,YouTube_Like_Ratio,TikTok_Like_Ratio,Spotify_Popularity_Norm,Platform_Success_Score
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,2024-04-26,QM24S2402528,1.0,415.4196,3.904709e+08,30716.0,196631588.0,...,4,26,4,2,Friday,31102.0,0.020328,0.122193,0.92,0.604504
1,Not Like Us,Not Like Us,Kendrick Lamar,2024-05-04,USUG12400910,2.0,415.4196,3.237039e+08,28113.0,174597137.0,...,5,4,5,2,Saturday,28479.0,0.029968,0.169068,0.92,0.615807
2,i like the way you kiss me,I like the way you kiss me,Artemas,2024-03-19,QZJ842400387,3.0,415.4196,6.013093e+08,54331.0,211607669.0,...,3,19,1,1,Tuesday,54829.0,0.018179,0.081669,0.92,0.595970
3,Flowers,Flowers - Single,Miley Cyrus,2023-01-12,USSM12209777,4.0,415.4196,2.031281e+09,269802.0,136569078.0,...,1,12,3,1,Thursday,270670.0,0.009698,0.073869,0.85,0.571713
4,Houdini,Houdini,Eminem,2024-05-31,USUG12403398,5.0,415.4196,1.070349e+08,7223.0,151469874.0,...,5,31,4,2,Friday,7592.0,0.047434,1.000000,0.88,0.773487


## ***`Save the cleaned dataset`***

In [13]:
df.to_csv('cleaned_spotify_songs_2024.csv', index=False)
print("✓ Cleaned dataset saved as 'cleaned_spotify_songs_2024.csv'")
# Display basic statistics of cleaned data
print("=" * 60)
print("BASIC STATISTICS OF KEY METRICS")
print("=" * 60)
df[['Spotify Streams', 'YouTube Views', 'TikTok Views', 'Track Score',
    'Platform_Success_Score']].describe().round(2)

✓ Cleaned dataset saved as 'cleaned_spotify_songs_2024.csv'
BASIC STATISTICS OF KEY METRICS


,Spotify Streams,YouTube Views,TikTok Views,Track Score,Platform_Success_Score
count,4.598000e+03,4.598000e+03,4.598000e+03,4598.00,4598.00
mean,4.764819e+08,3.745843e+08,8.241734e+08,41.72,0.28
std,5.602324e+08,6.361208e+08,2.361690e+09,36.88,0.09
min,1.071000e+03,9.130000e+02,1.900000e+01,19.40,0.03
25%,7.315308e+07,3.091328e+07,1.200000e+06,23.30,0.23
50%,2.512982e+08,1.283531e+08,1.267808e+08,29.90,0.26
75%,6.700540e+08,4.202808e+08,6.307454e+08,44.48,0.32
max,3.386428e+09,6.391486e+09,3.356549e+10,415.42,0.77
